# L6.5 — Evaluating LLM Output

Hands-on notebook for the lesson [`6-5-llm-eval.mdx`](../../llm-quest-theory/level-6/6-5-llm-eval.mdx).

> **Learning objectives**
> - Implement three families of eval metrics — exact/normalized match, n-gram overlap (ROUGE), and LLM-as-judge — and see where each one breaks.
> - Build a minimal eval harness: iterate over a test set, record per-question scores and aggregate them.
> - Run **pairwise preference** between two prompt variants and return the winner.
> - Observe the **hallucination** failure mode with a *faithfulness* metric over RAG answers.

## Connection to the theory
Covers **§1–§12** of the source `.mdx`. The LLM 'judge' is a local `flan-t5-base` — no paid APIs. The same harness applies to a frontier model.

In [1]:
# ---- Setup ----
import os, re, json, warnings, statistics
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 42
torch.manual_seed(SEED)
DEVICE = "cpu"

LLM_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(LLM_NAME).to(DEVICE)
model.eval()
print("judge / candidate model:", LLM_NAME)

judge / candidate model: google/flan-t5-base


In [2]:
@torch.no_grad()
def generate(prompt, max_new_tokens=100):
    ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).input_ids.to(DEVICE)
    out = model.generate(ids, max_new_tokens=max_new_tokens, num_beams=1, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True)

## 1. A tiny eval dataset
Short, mixed-difficulty QA items with expected answers. In production you aim for 100+ items.

In [3]:
TEST_SET = [
    {"id": "q1", "q": "What is the capital of France?",             "gold": "Paris"},
    {"id": "q2", "q": "What is 12 + 7?",                             "gold": "19"},
    {"id": "q3", "q": "Translate 'dog' to Spanish.",                  "gold": "perro"},
    {"id": "q4", "q": "Who wrote the play Hamlet?",                  "gold": "William Shakespeare"},
    {"id": "q5", "q": "What colour do you get when you mix blue and yellow?", "gold": "green"},
    {"id": "q6", "q": "How many sides does a hexagon have?",         "gold": "6"},
    {"id": "q7", "q": "What is the largest planet in our Solar System?", "gold": "Jupiter"},
    {"id": "q8", "q": "Who painted the Mona Lisa?",                  "gold": "Leonardo da Vinci"},
]
print(f"{len(TEST_SET)} items")

8 items


## 2. Metric 1 — Exact / Normalised match
Trivial, but the right starting point for fact-style QA.

In [4]:
def normalize(text):
    # Lowercase, strip punctuation and extra whitespace, remove articles.
    text = re.sub(r"\b(a|an|the)\b", " ", text.lower())
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def exact_match(pred, gold):
    return normalize(pred) == normalize(gold)

def contains_match(pred, gold):
    return normalize(gold) in normalize(pred)

# Small sanity tests
assert exact_match("The Paris.", "paris")
assert contains_match("The capital of France is Paris.", "Paris")
assert not exact_match("Paris is wonderful", "Paris")
print("normalize / exact / contains all sanity-check OK")

normalize / exact / contains all sanity-check OK


## 3. Metric 2 — ROUGE-1 / ROUGE-L from scratch
Traditional n-gram overlap scores. Useful for summarisation; loses when the model rephrases.

In [5]:
def _tokenise(text):
    return normalize(text).split()

def rouge_1_f1(pred, gold):
    p, g = _tokenise(pred), _tokenise(gold)
    if not p or not g: return 0.0
    p_set, g_set = set(p), set(g)
    common = p_set & g_set
    if not common: return 0.0
    prec = len(common) / len(p_set)
    rec  = len(common) / len(g_set)
    return 2 * prec * rec / (prec + rec)

def lcs_length(a, b):
    # Standard O(|a| * |b|) dynamic programming
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a)):
        for j in range(len(b)):
            dp[i+1][j+1] = dp[i][j] + 1 if a[i] == b[j] else max(dp[i][j+1], dp[i+1][j])
    return dp[-1][-1]

def rouge_l_f1(pred, gold):
    p, g = _tokenise(pred), _tokenise(gold)
    if not p or not g: return 0.0
    lcs = lcs_length(p, g)
    if lcs == 0: return 0.0
    prec = lcs / len(p)
    rec  = lcs / len(g)
    return 2 * prec * rec / (prec + rec)

# Sanity
print("ROUGE-1 identical :", rouge_1_f1("Paris France capital", "The capital of France is Paris"))
print("ROUGE-L identical :", rouge_l_f1("The capital is Paris", "The capital is Paris"))
print("ROUGE-L unrelated :", rouge_l_f1("I like apples", "The capital is Paris"))

ROUGE-1 identical : 0.7499999999999999
ROUGE-L identical : 1.0
ROUGE-L unrelated : 0.0


## 4. Metric 3 — LLM-as-judge (single-dimensional)
We ask a second model to rate correctness 1-5. With a tiny judge the scale is noisy — in production you'd use GPT-4 or Claude. The *pattern* is the same either way.

In [6]:
JUDGE_TEMPLATE = """You grade short answers.
Question: {q}
Reference answer: {gold}
Candidate answer: {pred}

Is the candidate correct? Reply with only YES or NO."""

def llm_judge(pred, gold, q):
    raw = generate(JUDGE_TEMPLATE.format(q=q, gold=gold, pred=pred), max_new_tokens=6)
    return raw.strip().upper().startswith("YES")

# Spot-check
print(llm_judge("Paris",              "Paris",       "What is the capital of France?"))
print(llm_judge("It is Paris, France","Paris",       "What is the capital of France?"))
print(llm_judge("London",             "Paris",       "What is the capital of France?"))

True
True
False


## 5. A minimal eval harness
Loop over the test set, capture predictions once, and score them with every metric.

In [7]:
def answer_prompt(q):
    return f"Answer briefly.\nQ: {q}\nA:"

def run_eval(test_set, answer_fn):
    results = []
    for item in test_set:
        pred = answer_fn(item["q"])
        row = {
            "id":            item["id"],
            "q":             item["q"],
            "gold":          item["gold"],
            "pred":          pred,
            "exact":         exact_match(pred, item["gold"]),
            "contains":      contains_match(pred, item["gold"]),
            "rouge_l":       rouge_l_f1(pred, item["gold"]),
            "judge_correct": llm_judge(pred, item["gold"], item["q"]),
        }
        results.append(row)
    return results

def aggregate(results):
    n = len(results)
    return {
        "exact@1"       : sum(r["exact"]         for r in results) / n,
        "contains@1"    : sum(r["contains"]      for r in results) / n,
        "rouge-L mean"  : statistics.mean(r["rouge_l"] for r in results),
        "judge accuracy": sum(r["judge_correct"] for r in results) / n,
    }

results = run_eval(TEST_SET, lambda q: generate(answer_prompt(q), max_new_tokens=20))
print(f"{'id':<4}{'exact':>6}{'cont':>6}{'rouge-L':>9}{'judge':>7}   pred")
print("-" * 80)
for r in results:
    print(f"{r['id']:<4}{int(r['exact']):>6}{int(r['contains']):>6}{r['rouge_l']:>9.2f}{int(r['judge_correct']):>7}   {r['pred'][:40]!r}")
print("\nAggregate:")
for k, v in aggregate(results).items():
    print(f"  {k:<18}{v:.3f}")

id   exact  cont  rouge-L  judge   pred
--------------------------------------------------------------------------------
q1       1     1     1.00      1   'Paris'
q2       0     0     0.00      0   '12 + 7'
q3       0     0     0.00      0   'Por qué?'
q4       0     0     0.00      1   'edward edward scott'
q5       0     0     0.00      0   '(D).'
q6       0     0     0.00      0   '4'
q7       0     0     0.00      0   '(d).'
q8       0     0     0.00      1   'franz heinrich schumann'

Aggregate:
  exact@1           0.125
  contains@1        0.125
  rouge-L mean      0.125
  judge accuracy    0.375


Each metric tells a slightly different story — *that's the point*. Exact match is strict; `contains` is lenient; ROUGE-L is continuous but rewards surface overlap; the judge is semantic but noisy.

## 6. Pairwise preference: variant A vs variant B
Two prompt styles, same test set. Ask the judge which answer is better per question, then aggregate.

In [8]:
VARIANT_A = "Answer briefly.\nQ: {q}\nA:"
VARIANT_B = "You are a helpful factual assistant. Answer with the shortest correct phrase.\nQ: {q}\nA:"

PAIRWISE_TEMPLATE = """You will compare two answers for the same question.
Question: {q}
Reference: {gold}
Answer A: {a}
Answer B: {b}

Which answer matches the reference better? Reply with only A, B, or TIE."""

def pairwise_winner(q, gold, a, b):
    raw = generate(PAIRWISE_TEMPLATE.format(q=q, gold=gold, a=a, b=b), max_new_tokens=4)
    raw = raw.strip().upper()
    if raw.startswith("A"): return "A"
    if raw.startswith("B"): return "B"
    return "TIE"

wins = {"A": 0, "B": 0, "TIE": 0}
for item in TEST_SET:
    a = generate(VARIANT_A.format(q=item["q"]), max_new_tokens=20)
    b = generate(VARIANT_B.format(q=item["q"]), max_new_tokens=20)
    # Alternate the order to cancel positional bias.
    order_ab = pairwise_winner(item["q"], item["gold"], a, b)
    order_ba = pairwise_winner(item["q"], item["gold"], b, a)
    # Translate the back-order vote
    order_ba_resolved = {"A": "B", "B": "A", "TIE": "TIE"}[order_ba]
    for vote in (order_ab, order_ba_resolved):
        wins[vote] += 1

print("pairwise votes (with order-flipping to cancel positional bias):", wins)

pairwise votes (with order-flipping to cancel positional bias): {'A': 9, 'B': 7, 'TIE': 0}


## 7. Faithfulness for RAG — does the answer come from the sources?
We adapt the judge to check whether each sentence in the answer is **supported** by a provided context. Real eval frameworks (RAGAs) formalise this.

In [9]:
FAITHFULNESS_TEMPLATE = """Is the claim supported by the context? Reply YES or NO only.
Context: {ctx}
Claim: {claim}
"""

def sentences(text):
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p for p in parts if p]

def faithfulness(answer, context):
    claims = sentences(answer)
    if not claims: return 1.0
    supported = 0
    for c in claims:
        raw = generate(FAITHFULNESS_TEMPLATE.format(ctx=context, claim=c), max_new_tokens=4)
        if raw.strip().upper().startswith("YES"):
            supported += 1
    return supported / len(claims)

CONTEXT = (
    "Paris is the capital of France and has about 2.1 million residents. "
    "The Eiffel Tower, built in 1889 for the World's Fair, is one of its most iconic landmarks."
)

GOOD = "Paris is the capital of France. The Eiffel Tower was built in 1889."
BAD  = "Paris is the capital of Italy. The Eiffel Tower was built in 1720."

print(f"faithful answer   : {faithfulness(GOOD, CONTEXT):.2f}   (expect high)")
print(f"unfaithful answer : {faithfulness(BAD,  CONTEXT):.2f}   (expect low)")

faithful answer   : 1.00   (expect high)
unfaithful answer : 0.00   (expect low)


## 8. Save results to a JSONL file
Every eval should produce a reproducible artefact — a JSONL of per-item scores that you can diff between runs.

In [10]:
OUT = os.environ.get("LLM_QUEST_DATA", "/tmp/data") + "/eval_results.jsonl"
with open(OUT, "w") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("wrote", OUT, "(", os.path.getsize(OUT), "bytes)")

wrote /tmp/data/eval_results.jsonl ( 1353 bytes)


## 9. Quick checks

In [11]:
# ROUGE and exact-match helpers are sane
assert exact_match("The Paris", "paris")
assert not exact_match("Paris, France", "Paris")
assert rouge_l_f1("same same", "same same") == 1.0
assert rouge_l_f1("nothing", "something else") == 0.0
# Aggregate produces every metric in [0, 1]
agg = aggregate(results)
assert all(0.0 <= v <= 1.0 for v in agg.values())
# Faithfulness discriminates between faithful and unfaithful answers
assert faithfulness(GOOD, CONTEXT) > faithfulness(BAD, CONTEXT)
# Pairwise evaluation did something (at least one non-TIE vote)
assert wins["A"] + wins["B"] >= 1
print("OK — three metric families + harness + pairwise + faithfulness all behave.")

OK — three metric families + harness + pairwise + faithfulness all behave.


## Reflection questions

1. Which of our four metrics is the most sensitive to minor rephrasings like "Paris." vs "The city of Paris"? Which is least sensitive?
2. Pairwise evaluation asks the judge twice (A-vs-B *and* B-vs-A) to cancel positional bias. What other bias would a single judge plausibly have, and how would you mitigate it?
3. Why does ROUGE-L score a valid summarisation as "low" when the summary uses different words than the reference? When would you still use ROUGE-L?
4. The faithfulness metric treats each sentence as a claim. On a 10-sentence answer, one unsupported sentence drops faithfulness to 0.9. Is that the right severity in practice?

## References
- Source theory: [`6-5-llm-eval.mdx`](../../llm-quest-theory/level-6/6-5-llm-eval.mdx)
- Next: [`6-6-doc-assistant-boss`](6-6-doc-assistant-boss.ipynb) — the Level 6 boss combining retrieval + generation + evaluation.